# Data Profiling — Urban Flow Analytics Taxi Dataset

**Run by:** [Praveen Madawalage]
**Date:** [2026/09/10]

Purpose: profile all 12 monthly CSVs before cleaning — check schema consistency,
dtypes, row counts, and date-range containment. This notebook's findings feed
directly into the cleaning decisions and the technical report's preprocessing section.

In [ ]:
import duckdb
import glob
import pandas as pd

files = sorted(glob.glob("../data/raw/*.csv"))  # adjust path if running from notebooks/
print(f"Found {len(files)} files")
for f in files:
    print(" -", f)

## 1. File Inventory & Schema Scan

Scanning each file's row count, column names, and dtypes using DuckDB
(lazy CSV scan — doesn't load full files into RAM).

In [ ]:
con = duckdb.connect()
profile_rows = []

for f in files:
    schema = con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{f}')").fetchdf()
    row_count = con.execute(f"SELECT COUNT(*) AS n FROM read_csv_auto('{f}')").fetchone()[0]

    profile_rows.append({
        "file": f,
        "n_rows": row_count,
        "n_cols": len(schema),
        "columns": tuple(schema["column_name"].tolist()),
        "dtypes": tuple(schema["column_type"].tolist()),
    })
    print(f"{f}: {row_count:,} rows, {len(schema)} cols")

profile_df = pd.DataFrame(profile_rows)
profile_df  

In [ ]:
# Save raw profile for the record / report appendix
profile_df.to_csv("../data/interim/file_schema_profile.csv", index=False)

## 2. Schema Consistency Check

- If unique column-name sets == 1 → all files match, safe to concatenate directly.
- If > 1 → inspect diffs below before merging (may be expected, e.g. a fee column
  introduced partway through the year per the data dictionary).

In [ ]:
unique_schemas = profile_df["columns"].nunique()
unique_dtypes = profile_df["dtypes"].nunique()
print(f"Unique column-name sets across {len(files)} files: {unique_schemas}")
print(f"Unique dtype sets across {len(files)} files: {unique_dtypes}")

if unique_schemas > 1:
    print("\n⚠️ SCHEMA MISMATCH DETECTED. Columns differ between files:")
    baseline = set(profile_df["columns"].iloc[0])
    for i, row in profile_df.iterrows():
        diff = set(row["columns"]).symmetric_difference(baseline)
        if diff:
            print(f"  {row['file']}: differs by {diff}")
else:
    print("\n✅ All files share the same column set.")

if unique_dtypes > 1:
    print("\n⚠️ DTYPE MISMATCH DETECTED — inspect per-column below.")
else:
    print("✅ All files share the same dtypes.")

## 3. Date Range Containment Check

Each file should contain pickups almost entirely within its labeled month.
Rows falling well outside that range suggest mislabeled or corrupted data.

In [ ]:
date_range_rows = []

for f in files:
    result = con.execute(f"""
        SELECT 
            MIN(pickup_timestamp) AS min_pickup, 
            MAX(pickup_timestamp) AS max_pickup,
            COUNT(*) AS total
        FROM read_csv_auto('{f}')
    """).fetchdf()
    row = result.to_dict("records")[0]
    row["file"] = f
    date_range_rows.append(row)
    print(f, row)

date_range_df = pd.DataFrame(date_range_rows)
date_range_df.to_csv("../data/interim/file_date_ranges.csv", index=False)
date_range_df

### Date range findings

*(Fill in after reviewing output above)*

- [ ] File [x]: date range [min–max] — matches expected month? Y/N
- [ ] Any files with stray out-of-month rows? How many / what % of file?

## 4. Decisions Going Into Cleaning

Summarize the concrete actions this profiling leads to, before writing the cleaning notebook:

1. ...
2. ...
3. ...

*(This section doubles as source material for the "Data Preprocessing" section of the final technical report.)*